# Generative AI and LLMs

**Case study:** Local document retrieval and grounded FAQ answering. Synthetic data only. Run cells in order.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("data") if Path("data/customers_synthetic.csv").exists() else Path(".")
customers = pd.read_csv(DATA / "customers_synthetic.csv")
orders = pd.read_csv(DATA / "orders_synthetic.csv", parse_dates=["order_date"])
print("Customers:", customers.shape, "Orders:", orders.shape)
display(customers.head())

## Data quality and exploratory checks

In [ ]:
assert customers.customer_id.is_unique
assert orders.order_id.is_unique
assert customers.churned.isin([0,1]).all()
print(customers.isna().sum().to_string())
print("Churn rate:", round(customers.churned.mean(),3))
customers.groupby("plan").churned.agg(["mean","count"]).plot(kind="bar",subplots=True,figsize=(9,3),title=["Churn rate","Customers"]); plt.tight_layout(); plt.show()

## Offline, API-free retrieval demo

This notebook teaches the retrieval and grounding components of RAG. It does **not** call an LLM or claim to generate model answers.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
docs={"policy-1":"Customers may request a refund within 30 days of purchase if the item is unused.","policy-2":"Standard shipping usually takes 3 to 5 business days after dispatch.","policy-3":"Support is available by email on weekdays from 9 AM to 5 PM.","policy-4":"Returned orders are excluded from completed sales revenue.","policy-5":"Premium plan members receive priority email support."}
ids=list(docs); texts=list(docs.values())
vectorizer=TfidfVectorizer(ngram_range=(1,2),stop_words="english")
matrix=vectorizer.fit_transform(texts)
def retrieve(question,k=2):
 scores=cosine_similarity(vectorizer.transform([question]),matrix).ravel()
 return [(ids[i],round(float(scores[i]),3),texts[i]) for i in np.argsort(scores)[::-1][:k]]
for item in retrieve("How long does standard shipping take?"):print(item)

## Evidence-first answering with abstention

In [ ]:
def evidence_answer(question,min_score=.10):
 hits=retrieve(question,k=1)
 if not hits or hits[0][1]<min_score:return {"answer":"Insufficient evidence in supplied documents.","sources":[]}
 source,score,passage=hits[0]
 return {"answer":passage,"sources":[source],"similarity":score}
for q in ["When can I request a refund?","What is the CEO salary?"]:
 print(q,"->",evidence_answer(q))
# Similarity thresholds are heuristic; manually review borderline cases.

## Small retrieval evaluation

In [ ]:
tests=[("How long is shipping?","policy-2"),("When can I get a refund?","policy-1"),("What are support hours?","policy-3"),("Do returns count as revenue?","policy-4")]
correct=sum(retrieve(q,k=1)[0][0]==expected for q,expected in tests)
print(f"Top-1 retrieval hit rate: {correct}/{len(tests)} = {correct/len(tests):.0%}")

## Reflection

1. What was your baseline? 2. What could go wrong in deployment? 3. Which results require human review?

Complete `Practice_Questions.md` before reading `Solutions.md`.